# 🌈 prism RL Environment — GRPO Training Demo
## Meta × PyTorch × Hugging Face OpenEnv Hackathon 2026

Prism is a **Scenario Generator System** built on top of **Meta's OpenEnv**. Unlike static benchmarks, Prism defines a **Scenario Space** through three fundamental **Failure Primitives**:
1. **Coordination Stress** (Multi-agent scaling)
2. **Atomic Failure** (System state integrity)
3. **Domain Shift** (Cross-domain transfer)

This notebook demonstrates how to train a model (like Qwen2.5-3B) using **GRPO** (Group Relative Policy Optimization) to navigate these primitives and develop "transaction-like discipline."

### 1. Install Dependencies

In [ ]:
!pip install openenv-core trl transformers torch wandb rich requests -q

### 2. Environment Configuration
Set the `ENV_URL` to your local backend or a running Hugging Face Space.

In [ ]:
# REPLACE with your URL if running on a Space
ENV_URL = "http://localhost:7860"

import requests
try:
    health = requests.get(f"{ENV_URL}/health").json()
    print(f"Environment status: {health}")
except:
    print("Backend not found! Ensure your environment is running.")

### 3. Step-by-Step Interaction
Watch how the **Dense Shaped Reward** reacts to agent actions. Note the **Atomic Health** component decaying as the agent works without a checkpoint.

In [ ]:
import requests, json

res = requests.post(f"{ENV_URL}/reset", json={
    "seed": 42,
    "options": {"task_domain": "debug", "agents": 2, "failure_rate": 0.2}
})
obs = res.json()["observation"]
episode_id = obs["episode_id"]
print(f"Episode started: {episode_id[:8]}...")

SEQUENCE = [
    {"tool": "research_web", "args": {"q": "analyse Python bug"}},
    {"tool": "write_code",   "args": {"path": "fix.py", "body": "def calculate_sum(items):\n    return sum(items)"}},
    {"tool": "checkpoint",   "args": {}},
    {"tool": "critique",     "args": {"target": "verified fix"}},
    {"tool": "finish",       "args": {"answer": "range(len(items)) fix applied"}},
]

rewards = []
for action in SEQUENCE:
    r = requests.post(f"{ENV_URL}/step", json={"action": action, "episode_id": episode_id})
    data = r.json()
    reward = data["reward"]
    rewards.append(reward)
    bd = data["info"]["reward_breakdown"]
    print(f"  {action['tool']:15} reward={reward:.4f}  "
          f"progress={bd['progress_delta']:.2f}  "
          f"atomic={bd['atomic_health']:.2f}  "
          f"coord={bd['coord_efficiency']:.2f}")

print(f"\nTotal episode reward: {sum(rewards):.4f}")

### 4. Training Simulation (Evidence Generator)
This loop simulates a training run and plots the resulting reward curves.

In [ ]:
import json, time, os
import matplotlib.pyplot as plt

N_EPISODES = 30
all_rewards = []
rolling_rewards = []
rolling = 0.3

for ep in range(1, N_EPISODES + 1):
    res = requests.post(f"{ENV_URL}/reset", json={
        "seed": 200 + ep,
        "options": {"task_domain": "debug", "agents": 2, "failure_rate": 0.1}
    })
    eid = res.json()["observation"]["episode_id"]

    ep_total = 0.0
    for action in SEQUENCE:
        r = requests.post(f"{ENV_URL}/step",
                          json={"action": action, "episode_id": eid})
        ep_total += r.json().get("reward", 0.0)

    rolling = 0.1 * ep_total + 0.9 * rolling
    all_rewards.append(ep_total)
    rolling_rewards.append(rolling)
    if ep % 5 == 0: print(f"[{ep:3d}/{N_EPISODES}] rolling={rolling:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(all_rewards, alpha=0.3, color='blue', label='Episode Reward')
plt.plot(rolling_rewards, color='blue', linewidth=2, label='Rolling Average (Policy Progress)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('prism RL Training - Behavioral Learning Curve')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

### 5. Multi-Model Tournament
Demonstrating that Prism is **Model-Agnostic**. We can switch models mid-flight and compare their reliability signatures.

In [ ]:
print("Visit the Prism Dashboard at http://localhost:7860 to see model tournaments in real-time!")